***Preparação***

In [55]:
import numpy as np
import time

def generate_data(num_samples=400):
    """Gera dados com a regra condicional de temperatura."""
    X, y = [], []
    for _ in range(num_samples):
        temp, pressure, vibration = np.random.rand(), np.random.rand(), np.random.rand()
        label = 0 # Reprovado por padrão
        
        # A regra de exceção: temperatura na zona de perigo -> reprovação imediata
        if 0.7 <= temp <= 0.9:
            label = 0
        # A regra normal: só checada se a primeira for falsa
        elif pressure > 0.6 and vibration < 0.4:
            label = 1
            
        X.append([temp, pressure, vibration])
        y.append([label])
    return np.array(X), np.array(y)

X_train, y_train = generate_data()

In [57]:
def generate_initial_weights(input_size=3, h1_size=5, h2_size=4, output_size=1, seed=42):
    """Gera pesos/vieses iniciais determinísticos para comparar modelos."""
    rng = np.random.default_rng(seed)
    # Camada H1
    weights_h1 = rng.normal(0, 0.1, size=(input_size, h1_size))
    bias_h1 = np.zeros(h1_size)
    # Camada H2 (densa)
    weights_h2 = rng.normal(0, 0.1, size=(h1_size, h2_size))
    bias_h2 = np.zeros(h2_size)
    # Saída
    weights_out = rng.normal(0, 0.1, size=(h2_size, output_size))
    bias_out = np.zeros(output_size)
    return {
        'weights_h1': weights_h1,
        'bias_h1': bias_h1,
        'weights_h2': weights_h2,
        'bias_h2': bias_h2,
        'weights_out': weights_out,
        'bias_out': bias_out,
    }

initial_weights = generate_initial_weights()
initial_weights

{'weights_h1': array([[ 0.03047171, -0.10399841,  0.07504512,  0.09405647, -0.19510352],
        [-0.13021795,  0.01278404, -0.03162426, -0.00168012, -0.08530439],
        [ 0.0879398 ,  0.07777919,  0.00660307,  0.11272412,  0.04675093]]),
 'bias_h1': array([0., 0., 0., 0., 0.]),
 'weights_h2': array([[-0.08592925,  0.03687508, -0.09588826,  0.08784503],
        [-0.00499259, -0.01848624, -0.06809295,  0.12225413],
        [-0.01545295, -0.04283278, -0.03521336,  0.05323092],
        [ 0.03654441,  0.04127326,  0.0430821 ,  0.21416476],
        [-0.0406415 , -0.05122427, -0.08137727,  0.06159794]]),
 'bias_h2': array([0., 0., 0., 0.]),
 'weights_out': array([[ 0.11289723],
        [-0.01139475],
        [-0.08401565],
        [-0.08244812]]),
 'bias_out': array([0.])}

***Modelo de rede neural multi perceptron tradicional***

In [58]:
class StandardMLP:
    """
    Uma Rede Neural Multilayer Perceptron (MLP) tradicional.
    A arquitetura é 3 (entrada) -> 5 (oculta 1) -> 4 (oculta 2) -> 1 (saída).
    Esta rede não possui nenhuma lógica de comporta customizada.
    """
    def __init__(self, initial_weights, input_size=3, h1_size=5, h2_size=4, output_size=1):
        # Os pesos e vieses são armazenados como matrizes e vetores NumPy.
        # Inicializamos com valores pequenos e aleatórios para quebrar a simetria.
        self.weights_h1 = initial_weights["weights_h1"]
        self.bias_h1 = initial_weights["bias_h1"]
        
        # A camada H2 é uma camada densa padrão, assim como a H1.
        self.weights_h2 = initial_weights["weights_h2"]
        self.bias_h2 = initial_weights["bias_h2"]
        
        self.weights_out = initial_weights["weights_out"]
        self.bias_out = initial_weights["bias_out"]
        
        print("Standard MLP (3-5-4-1) criada.")

    def _sigmoid(self, x):
        """Função de ativação sigmoide. Coloca os valores entre 0 e 1."""
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        """Derivada da sigmoide, necessária para o backpropagation."""
        return x * (1 - x)

    def predict(self, inputs):
        """
        Realiza o 'Forward Pass': calcula a predição da rede para uma dada entrada.
        """
        # Da entrada para a camada oculta 1
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)
        
        # Da camada oculta 1 para a camada oculta 2
        self.h2_output = self._sigmoid(np.dot(self.h1_output, self.weights_h2) + self.bias_h2)
        
        # Da camada oculta 2 para a camada de saída
        final_output = self._sigmoid(np.dot(self.h2_output, self.weights_out) + self.bias_out)
        
        return final_output

    def train(self, X, y, epochs=10000, learning_rate=0.1):
        """
        Executa o treinamento da rede usando o algoritmo de backpropagation.
        """
        print(f"Iniciando treinamento por {epochs} épocas...")
        start_time = time.time()
        
        for epoch in range(epochs):
            total_error = 0
            for inputs, expected in zip(X, y):
                # --- 1. Forward Pass ---
                # A predição é calculada para obter o erro.
                final_output = self.predict(inputs)

                # --- 2. Backward Pass (Cálculo dos Gradientes) ---
                # O erro é a diferença entre o esperado и o obtido.
                error = expected - final_output
                total_error += np.sum(error**2)

                # Calcula o gradiente (delta) para cada camada, de trás para frente.
                d_output = error * self._sigmoid_derivative(final_output)
                
                error_h2 = d_output.dot(self.weights_out.T)
                d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                
                error_h1 = d_h2.dot(self.weights_h2.T)
                d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                # --- 3. Atualização dos Pesos e Vieses ---
                # Ajusta os parâmetros da rede na direção que minimiza o erro.
                self.weights_out += self.h2_output.reshape(-1, 1) * d_output * learning_rate
                self.bias_out += d_output * learning_rate
                
                self.weights_h2 += self.h1_output.reshape(-1, 1) * d_h2 * learning_rate
                self.bias_h2 += d_h2 * learning_rate
                
                self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                self.bias_h1 += d_h1 * learning_rate
            
            if (epoch + 1) % 500 == 0:
                print(f"  Época {epoch + 1}/{epochs}, Erro Total: {total_error:.10f}")

        end_time = time.time()
        print(f"Treinamento concluído em {end_time - start_time:.2f} segundos.\n")


# Instanciar a rede neural tradicional

standard_mlp = StandardMLP(initial_weights=initial_weights)

print("--- Iniciando Treinamento ---")
standard_mlp.train(X_train, y_train)
print("--- Treinamento Concluído ---\n")

print("="*40)
print("      AVALIAÇÃO DO MODELO TRADICIONAL")
print("="*40)

# Definir os mesmos casos de teste
test_1 = np.array([0.2, 0.8, 0.1]) # Aprovação Esperada: 1
test_2 = np.array([0.3, 0.2, 0.2]) # Reprovação Esperada: 0
test_3 = np.array([0.8, 0.9, 0.1]) # Reprovação por Temperatura (CASO CRÍTICO): 0

tests = [test_1, test_2, test_3]
expected_results = [1, 0, 0]
test_names = ["Aprovação Normal", "Reprovação Normal", "Reprovação por Temperatura (Crítico)"]

for i, test_case in enumerate(tests):
    print(f"\n--- Caso de Teste: {test_names[i]} ---")
    print(f"Entrada: {test_case}, Resultado Esperado: {expected_results[i]}")

    # Obter a predição do modelo tradicional
    prediction = standard_mlp.predict(test_case)
    resultado_final = 'Aprovado' if prediction > 0.5 else 'Reprovado'
    print(f"  Predição da Standard MLP: {prediction[0]:.10f} -> {resultado_final}")

Standard MLP (3-5-4-1) criada.
--- Iniciando Treinamento ---
Iniciando treinamento por 10000 épocas...
  Época 500/10000, Erro Total: 24.0892185238
  Época 1000/10000, Erro Total: 15.7922908739
  Época 1500/10000, Erro Total: 8.5284437671
  Época 2000/10000, Erro Total: 7.4616602187
  Época 2500/10000, Erro Total: 4.7890003671
  Época 3000/10000, Erro Total: 3.1707906293
  Época 3500/10000, Erro Total: 2.5348076353
  Época 4000/10000, Erro Total: 1.6614885008
  Época 4500/10000, Erro Total: 1.3565148999
  Época 5000/10000, Erro Total: 1.0555874399
  Época 5500/10000, Erro Total: 0.6251250569
  Época 6000/10000, Erro Total: 0.1289589471
  Época 6500/10000, Erro Total: 0.0764774888
  Época 7000/10000, Erro Total: 0.0566905458
  Época 7500/10000, Erro Total: 0.0453300661
  Época 8000/10000, Erro Total: 0.0377751006
  Época 8500/10000, Erro Total: 0.0323406380
  Época 9000/10000, Erro Total: 0.0282301424
  Época 9500/10000, Erro Total: 0.0250086426
  Época 10000/10000, Erro Total: 0.022415

***Modelo de rede neural multi perceptron com comporta forte SmartMLP***

In [59]:
class SmartMLP:
    """
    MLP com opção de ignorar parcial ou totalmente a camada oculta H2.
    Quando use_h2=False, a saída conecta diretamente H1 -> Saída por pesos de bypass.
    Quando use_h2=True, é possível ativar um subconjunto dos neurônios de H2 via máscara (h2_mask).
    Durante o treinamento (try_both=True), testamos AUTOMATICAMENTE:
      - bypass (0 neurônios H2)
      - TODAS as combinações de 1..h2_size neurônios ativos em H2,
    mantendo a configuração (máscara) que produzir o menor erro total.
    """
    def __init__(self, initial_weights, input_size=3, h1_size=5, h2_size=4, output_size=1, use_h2=True):
        # Pesos/vieses das camadas tradicionais (fornecidos externamente)
        self.weights_h1 = initial_weights["weights_h1"]
        self.bias_h1 = initial_weights["bias_h1"]
        self.weights_h2 = initial_weights["weights_h2"]
        self.bias_h2 = initial_weights["bias_h2"]
        self.weights_out = initial_weights["weights_out"]
        self.bias_out = initial_weights["bias_out"]

        # Pesos/vieses para o caminho de bypass (H1 -> Saída). Inicializamos com zeros para
        # garantir determinismo. Eles serão ajustados se use_h2=False vencer no treinamento.
        import numpy as _np
        self.weights_bypass_out = _np.zeros((h1_size, output_size))
        self.bias_bypass_out = _np.zeros(output_size)

        # Máscara de H2 (1.0 = ativo, 0.0 = inativo). Por padrão, todos ativos.
        self.h2_mask = _np.ones(h2_size)

        self.use_h2 = bool(use_h2)

    def _sigmoid(self, x):
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        return x * (1 - x)

    def predict(self, inputs, use_h2=None):
        """
        Forward pass que respeita a flag de uso da H2 e uma possível máscara de neurônios.
        - Se use_h2=True: Entrada -> H1(sigmoid) -> H2(sigmoid & máscara) -> Out(sigmoid)
        - Se use_h2=False: Entrada -> H1(sigmoid) -> Out_bypass(sigmoid)
        """
        if use_h2 is None:
            use_h2 = self.use_h2

        # Camada H1
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)

        if use_h2:
            # Camada H2
            self.h2_output = self._sigmoid(np.dot(self.h1_output, self.weights_h2) + self.bias_h2)
            # Aplicar máscara de H2 (neurônios inativos não contribuem)
            self.h2_used = self.h2_output * self.h2_mask
            # Saída final via H2 mascarada
            final_output = self._sigmoid(np.dot(self.h2_used, self.weights_out) + self.bias_out)
        else:
            # Ignoramos H2 e usamos o caminho de bypass H1 -> Saída
            final_output = self._sigmoid(np.dot(self.h1_output, self.weights_bypass_out) + self.bias_bypass_out)

        return final_output

    def _snapshot(self):
        """Cria um snapshot dos pesos atuais para comparar cenários."""
        return {
            'weights_h1': self.weights_h1.copy(),
            'bias_h1': self.bias_h1.copy(),
            'weights_h2': self.weights_h2.copy(),
            'bias_h2': self.bias_h2.copy(),
            'weights_out': self.weights_out.copy(),
            'bias_out': self.bias_out.copy(),
            'weights_bypass_out': self.weights_bypass_out.copy(),
            'bias_bypass_out': self.bias_bypass_out.copy(),
            'h2_mask': self.h2_mask.copy(),
            'use_h2': bool(self.use_h2),
        }

    def _restore(self, snap):
        """Restaura pesos a partir de um snapshot."""
        self.weights_h1 = snap['weights_h1'].copy()
        self.bias_h1 = snap['bias_h1'].copy()
        self.weights_h2 = snap['weights_h2'].copy()
        self.bias_h2 = snap['bias_h2'].copy()
        self.weights_out = snap['weights_out'].copy()
        self.bias_out = snap['bias_out'].copy()
        self.weights_bypass_out = snap['weights_bypass_out'].copy()
        self.bias_bypass_out = snap['bias_bypass_out'].copy()
        self.h2_mask = snap['h2_mask'].copy()
        self.use_h2 = bool(snap['use_h2'])

    def _train_single(self, X, y, epochs, learning_rate, use_h2_flag, h2_mask=None):
        """
        Treina um único cenário e retorna o erro total ao final.
        - use_h2_flag=True: utiliza H2 com máscara (se fornecida).
        - use_h2_flag=False: utiliza caminho de bypass (H1->Saída).
        """
        import numpy as _np
        self.use_h2 = use_h2_flag
        if use_h2_flag:
            if h2_mask is None:
                # Se não foi passada máscara, ativa todos os neurônios de H2
                self.h2_mask = _np.ones(self.weights_h2.shape[1])
            else:
                self.h2_mask = _np.array(h2_mask, dtype=float)
        total_error = 0.0
        for epoch in range(epochs):
            epoch_error = 0.0
            for inputs, expected in zip(X, y):
                final_output = self.predict(inputs, use_h2=use_h2_flag)
                error = expected - final_output
                epoch_error += float(np.sum(error ** 2))

                d_output = error * self._sigmoid_derivative(final_output)

                if use_h2_flag:
                    # Backprop com H2 (respeitando a máscara)
                    error_h2 = d_output.dot(self.weights_out.T)
                    d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                    # Zerar gradientes dos neurônios inativos de H2
                    d_h2 = d_h2 * self.h2_mask

                    error_h1 = d_h2.dot(self.weights_h2.T)
                    d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                    # Atualizações (usar h2_used para respeitar máscara no gradiente de saída)
                    self.weights_out += self.h2_used.reshape(-1, 1) * d_output * learning_rate
                    self.bias_out += d_output * learning_rate

                    self.weights_h2 += self.h1_output.reshape(-1, 1) * d_h2 * learning_rate
                    self.bias_h2 += d_h2 * learning_rate

                    self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                    self.bias_h1 += d_h1 * learning_rate
                else:
                    # Backprop sem H2 (via caminho de bypass)
                    error_h1 = d_output.dot(self.weights_bypass_out.T)
                    d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)

                    # Atualizações para bypass e H1
                    self.weights_bypass_out += self.h1_output.reshape(-1, 1) * d_output * learning_rate
                    self.bias_bypass_out += d_output * learning_rate

                    self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                    self.bias_h1 += d_h1 * learning_rate

            # Log de treinamento a cada 500 épocas e na última época
            if ((epoch + 1) % 500 == 0) or ((epoch + 1) == epochs):
                if use_h2_flag:
                    mask_list = self.h2_mask.astype(int).tolist()
                    false_idx = [i for i, v in enumerate(mask_list) if v == 0]
                else:
                    false_idx = list(range(self.weights_h2.shape[1]))
                print(f"Época {epoch+1}/{epochs}, use_h2={use_h2_flag}, h2_false={false_idx}, Erro: {epoch_error:.10f}")

            total_error = epoch_error
        return total_error

    def train(self, X, y, epochs=2000, learning_rate=0.1, try_both=True):
        """
        Executa o treinamento.
        - Se try_both=True: testa bypass (0 H2) e TODAS as combinações possíveis de neurônios de H2 (1..h2_size),
          escolhe o menor erro final e mantém os pesos e a máscara referentes ao melhor cenário.
        - Se try_both=False: treina apenas no modo atual self.use_h2 e máscara atual self.h2_mask.
        """
        import time as _time
        import itertools as _itertools
        import numpy as _np
        start = _time.time()

        if not try_both:
            final_error = self._train_single(X, y, epochs, learning_rate, self.use_h2, h2_mask=self.h2_mask)
            print(f"Treinamento concluído (use_h2={self.use_h2}, mask={self.h2_mask.astype(int).tolist()}) com erro final: {final_error:.4f}")
            print(f"Tempo: {_time.time() - start:.2f}s")
            return {'use_h2': self.use_h2, 'final_error': final_error, 'h2_mask': self.h2_mask.copy()}

        # Snapshot inicial para replicar condições em todos os cenários
        base = self._snapshot()

        results = []

        # Cenário 0: bypass (use_h2=False)
        self._restore(base)
        err_bypass = self._train_single(X, y, epochs, learning_rate, use_h2_flag=False)
        snap_bypass = self._snapshot()
        results.append(('bypass', err_bypass, None, snap_bypass))

        # Demais cenários: todas as combinações de 1..h2_size neurônios ativos em H2
        h2_size = self.weights_h2.shape[1]
        indices = list(range(h2_size))
        for k in range(h2_size, 0, -1):
            for combo in _itertools.combinations(indices, k):
                mask = _np.zeros(h2_size, dtype=float)
                mask[list(combo)] = 1.0
                self._restore(base)
                err = self._train_single(X, y, epochs, learning_rate, use_h2_flag=True, h2_mask=mask)
                snap = self._snapshot()
                results.append((f'h2_k={k}', err, mask.copy(), snap))

        # Escolha do melhor cenário
        best = min(results, key=lambda t: t[1])
        label, chosen_err, chosen_mask, chosen_snap = best

        self._restore(chosen_snap)
        if label == 'bypass':
            self.use_h2 = False
            self.h2_mask = _np.zeros(h2_size)
        else:
            self.use_h2 = True
            self.h2_mask = chosen_mask.copy()

        print("Resultados SmartMLP:")
        print(f"  bypass (use_h2=False) -> erro final: {err_bypass:.4f}")
        # Opcional: imprimir um resumo de algumas combinações testadas
        # Aqui, mostramos apenas as melhores 3 combinações com H2 para evitar poluição de logs
        only_h2 = [(m, e) for (lab, e, m, s) in results if lab != 'bypass']
        only_h2_sorted = sorted(only_h2, key=lambda t: t[1])
        for i, (m, e) in enumerate(only_h2_sorted[:3]):
            print(f"  h2_mask_top{i+1}={m.astype(int).tolist()} -> erro final: {e:.10f}")
        print(f"  Escolhido: use_h2={self.use_h2}, mask={self.h2_mask.astype(int).tolist()} (erro {chosen_err:.10f})")
        print(f"Tempo total: {_time.time() - start:.2f}s")
        return {
            'use_h2': self.use_h2,
            'final_error': chosen_err,
            'h2_mask': self.h2_mask.copy(),
            'err_bypass': err_bypass,
        }

smart_mlp = SmartMLP(initial_weights=initial_weights)
# Treinar a rede
print("--- Iniciando Treinamento ---")
smart_mlp.train(X_train, y_train)
print("--- Treinamento Concluído ---\n")

print("="*40)
print("      AVALIAÇÃO DO MODELO GATE")
print("="*40)

# Definir os mesmos casos de teste
test_1 = np.array([0.2, 0.8, 0.1]) # Aprovação Esperada: 1
test_2 = np.array([0.3, 0.2, 0.2]) # Reprovação Esperada: 0
test_3 = np.array([0.8, 0.9, 0.1]) # Reprovação por Temperatura (CASO CRÍTICO): 0

tests = [test_1, test_2, test_3]
expected_results = [1, 0, 0]
test_names = ["Aprovação Normal", "Reprovação Normal", "Reprovação por Temperatura (Crítico)"]

for i, test_case in enumerate(tests):
    print(f"\n--- Caso de Teste: {test_names[i]} ---")
    print(f"Entrada: {test_case}, Resultado Esperado: {expected_results[i]}")

    # Obter a predição do modelo tradicional
    prediction = smart_mlp.predict(test_case)
    resultado_final = 'Aprovado' if prediction > 0.5 else 'Reprovado'
    print(f"  Predição da Standard MLP: {prediction[0]:.10f} -> {resultado_final}")

--- Iniciando Treinamento ---
Época 500/2000, use_h2=False, h2_false=[0, 1, 2, 3], Erro: 3.3809986566
Época 1000/2000, use_h2=False, h2_false=[0, 1, 2, 3], Erro: 2.2068203074
Época 1500/2000, use_h2=False, h2_false=[0, 1, 2, 3], Erro: 1.6988294422
Época 2000/2000, use_h2=False, h2_false=[0, 1, 2, 3], Erro: 1.3863702049
Época 500/2000, use_h2=True, h2_false=[], Erro: 0.0202835896
Época 1000/2000, use_h2=True, h2_false=[], Erro: 0.0185009118
Época 1500/2000, use_h2=True, h2_false=[], Erro: 0.0169889481
Época 2000/2000, use_h2=True, h2_false=[], Erro: 0.0156911416
Época 500/2000, use_h2=True, h2_false=[3], Erro: 0.5880931311
Época 1000/2000, use_h2=True, h2_false=[3], Erro: 0.3116277699
Época 1500/2000, use_h2=True, h2_false=[3], Erro: 0.1753380283
Época 2000/2000, use_h2=True, h2_false=[3], Erro: 0.1136608961
Época 500/2000, use_h2=True, h2_false=[2], Erro: 0.0316037463
Época 1000/2000, use_h2=True, h2_false=[2], Erro: 0.0273129187
Época 1500/2000, use_h2=True, h2_false=[2], Erro: 0.0241

***Modelo de rede neural multi perceptron com comporta forte SmartComMLP***

In [60]:
class SmartComMLP:
    """
    MLP 3-5-4-1 com duas fases de treino:
    - Prévia: 50 épocas com taxa de aprendizado variável (multiplicadores: 5x, 4x, 3x, 2x, 1x).
    - Final: 3000 épocas com learning_rate fixo = 0.1, usando a melhor combinação de neurônios H2 da prévia.
    
    Objetivo: analisar a derivada aproximada do erro por época na prévia e selecionar a combinação
    (máscara de H2 ou bypass) que produziu o menor erro final, então refinar com treino fixo.
    """
    def __init__(self, initial_weights, input_size=3, h1_size=5, h2_size=4, output_size=1):
        # Pesos/vieses iniciais fornecidos externamente para determinismo.
        self.weights_h1 = initial_weights['weights_h1']
        self.bias_h1 = initial_weights['bias_h1']
        self.weights_h2 = initial_weights['weights_h2']
        self.bias_h2 = initial_weights['bias_h2']
        self.weights_out = initial_weights['weights_out']
        self.bias_out = initial_weights['bias_out']
        # Pesos/vieses de bypass (H1 -> Saída), iniciados em zero para determinismo.
        self.weights_bypass_out = np.zeros((h1_size, output_size))
        self.bias_bypass_out = np.zeros(output_size)
        # Máscara de H2 (por padrão todos ativos) e flag de uso da H2.
        self.h2_mask = np.ones(h2_size)
        self.use_h2 = True

    def _sigmoid(self, x):
        """Função de ativação sigmoide.
        Comentário: limita valores entre 0 e 1 para estabilidade.
        """
        return 1 / (1 + np.exp(-x))

    def _sigmoid_derivative(self, x):
        """Derivada da sigmoide, usada no backpropagation.
        Comentário: calcula x*(1-x), assumindo x como saída da sigmoide.
        """
        return x * (1 - x)

    def predict(self, inputs, use_h2=None):
        """Forward pass com opção de usar/bypassar H2.
        Comentário: aplica máscara em H2 quando ativa e caminho de bypass quando desativa.
        """
        if use_h2 is None:
            use_h2 = self.use_h2
        # Entrada -> H1
        self.h1_output = self._sigmoid(np.dot(inputs, self.weights_h1) + self.bias_h1)
        if use_h2:
            # H1 -> H2 (com máscara)
            self.h2_output = self._sigmoid(np.dot(self.h1_output, self.weights_h2) + self.bias_h2)
            self.h2_used = self.h2_output * self.h2_mask
            # H2 -> Saída
            final_output = self._sigmoid(np.dot(self.h2_used, self.weights_out) + self.bias_out)
        else:
            # Bypass H2: H1 -> Saída
            final_output = self._sigmoid(np.dot(self.h1_output, self.weights_bypass_out) + self.bias_bypass_out)
        return final_output

    def _snapshot(self):
        """Cria um snapshot completo do estado atual (pesos, vieses, máscara e modo).
        Comentário: usado para restaurar exatamente o cenário vencedor após a prévia.
        """
        return {
            'weights_h1': self.weights_h1.copy(),
            'bias_h1': self.bias_h1.copy(),
            'weights_h2': self.weights_h2.copy(),
            'bias_h2': self.bias_h2.copy(),
            'weights_out': self.weights_out.copy(),
            'bias_out': self.bias_out.copy(),
            'weights_bypass_out': self.weights_bypass_out.copy(),
            'bias_bypass_out': self.bias_bypass_out.copy(),
            'h2_mask': self.h2_mask.copy(),
            'use_h2': bool(self.use_h2),
        }

    def _restore(self, snap):
        """Restaura completamente o estado a partir de um snapshot.
        Comentário: garante reproducibilidade entre cenários avaliados.
        """
        self.weights_h1 = snap['weights_h1'].copy()
        self.bias_h1 = snap['bias_h1'].copy()
        self.weights_h2 = snap['weights_h2'].copy()
        self.bias_h2 = snap['bias_h2'].copy()
        self.weights_out = snap['weights_out'].copy()
        self.bias_out = snap['bias_out'].copy()
        self.weights_bypass_out = snap['weights_bypass_out'].copy()
        self.bias_bypass_out = snap['bias_bypass_out'].copy()
        self.h2_mask = snap['h2_mask'].copy()
        self.use_h2 = bool(snap['use_h2'])

    def _lr_schedule_multiplier(self, epoch_one_based: int) -> float:
        """
        Agenda de taxa de aprendizado por época (100 épocas):
        - Épocas 1-20: 5x
        - Épocas 21-40: 4x
        - Épocas 41-60: 3x
        - Épocas 61-80: 2x
        - Épocas 81-100: 1x
        """
        if 1 <= epoch_one_based <= 10:
            return 0.1
        elif 11 <= epoch_one_based <= 20:
            return 0.2
        elif 21 <= epoch_one_based <= 30:
            return 0.3
        elif 31 <= epoch_one_based <= 40:
            return 0.4
        else:
            return 0.5
    def _train_single_variable_lr(self, X, y, epochs=25, base_lr=0.1, use_h2_flag=True, h2_mask=None):
        """Treina um cenário por 50 épocas com LR variável segundo a agenda.
        Comentário: coleta histórico de erro e derivada aproximada (diferenças sucessivas).
        """
        import numpy as _np
        self.use_h2 = use_h2_flag
        if use_h2_flag:
            self.h2_mask = _np.ones(self.weights_h2.shape[1]) if h2_mask is None else _np.array(h2_mask, dtype=float)
        err_history = []
        for epoch in range(epochs):
            mult = self._lr_schedule_multiplier(epoch + 1)
            learning_rate = base_lr * mult
            epoch_error = 0.0
            for inputs, expected in zip(X, y):
                final_output = self.predict(inputs, use_h2=self.use_h2)
                error = expected - final_output
                epoch_error += float(_np.sum(error ** 2))
                d_output = error * self._sigmoid_derivative(final_output)
                if self.use_h2:
                    error_h2 = d_output.dot(self.weights_out.T)
                    d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                    d_h2 = d_h2 * self.h2_mask
                    error_h1 = d_h2.dot(self.weights_h2.T)
                    d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)
                    self.weights_out += self.h2_used.reshape(-1, 1) * d_output * learning_rate
                    self.bias_out += d_output * learning_rate
                    self.weights_h2 += self.h1_output.reshape(-1, 1) * d_h2 * learning_rate
                    self.bias_h2 += d_h2 * learning_rate
                    self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                    self.bias_h1 += d_h1 * learning_rate
                else:
                    error_h1 = d_output.dot(self.weights_bypass_out.T)
                    d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)
                    self.weights_bypass_out += self.h1_output.reshape(-1, 1) * d_output * learning_rate
                    self.bias_bypass_out += d_output * learning_rate
                    self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                    self.bias_h1 += d_h1 * learning_rate
            # Log por época no formato solicitado (a cada 25 épocas)
            if self.use_h2:
                h2_false = [i for i, v in enumerate(self.h2_mask.tolist()) if v == 0.0]
            else:
                h2_false = list(range(self.weights_h2.shape[1]))
            if (epoch + 1) % 100 == 0 or (epoch + 1) == epochs:
                print(f"Época {epoch+1}/{epochs}, use_h2={self.use_h2}, h2_false={h2_false}, Erro: {epoch_error:.10f}")
            err_history.append(epoch_error)
        return err_history[-1], err_history

    def _preselect_best(self, X, y, epochs=25, base_lr=0.1):
        """Explora bypass e todas as máscaras possíveis de H2, escolhe menor erro final da prévia.
        Comentário: usa snapshots para isolar cenários e comparar de forma justa.
        """
        import itertools as _itertools
        import numpy as _np
        base = self._snapshot()
        results = []
        # Cenário bypass
        self._restore(base)
        err_bypass, hist_bypass = self._train_single_variable_lr(X, y, epochs, base_lr, use_h2_flag=False)
        snap_bypass = self._snapshot()
        h2_size_b = self.weights_h2.shape[1]
        results.append(('bypass', err_bypass, None, snap_bypass, hist_bypass))
        # Demais cenários com H2
        h2_size = self.weights_h2.shape[1]
        indices = list(range(h2_size))
        for k in range(h2_size, 0, -1):
            for combo in _itertools.combinations(indices, k):
                mask = _np.zeros(h2_size, dtype=float)
                mask[list(combo)] = 1.0
                self._restore(base)
                err, hist = self._train_single_variable_lr(X, y, epochs, base_lr, use_h2_flag=True, h2_mask=mask)
                h2_false = [i for i, v in enumerate(mask.tolist()) if v == 0.0]
                snap = self._snapshot()
                results.append((f'h2_k={k}', err, mask.copy(), snap, hist))
        best = min(results, key=lambda t: t[1])
        print(" Resultados SmartComMLP: ")
        bypass_err = [r[1] for r in results if r[0] == 'bypass'][0]
        print(f"   bypass (use_h2=False) -> erro final: {bypass_err:.4f}")
        h2_results = [r for r in results if r[0] != 'bypass']
        h2_sorted = sorted(h2_results, key=lambda t: t[1])
        for idx, (_lbl, err, mask, _snap, _hist) in enumerate(h2_sorted[:3], start=1):
            mask_list = [int(v) for v in mask.tolist()]
            print(f"   h2_mask_top{idx}={mask_list} -> erro final: {err:.10f}")
        chosen_use_h2 = best[0] != 'bypass'
        chosen_mask_list = None if not chosen_use_h2 else [int(v) for v in best[2].tolist()]
        print(f"   Escolhido: use_h2={chosen_use_h2}, mask={chosen_mask_list} (erro {best[1]:.10f})")
        return best  # (label, err_final, mask_or_None, snapshot, err_history)

    def _train_final_fixed(self, X, y, epochs=10000, learning_rate=0.1, use_h2_flag=None):
        """Treina com LR fixo (0.1) por 3000 épocas usando a configuração escolhida.
        Comentário: aplica backprop padrão respeitando máscara (se H2 ativa) ou bypass.
        """
        import numpy as _np
        if use_h2_flag is None:
            use_h2_flag = self.use_h2
        self.use_h2 = use_h2_flag
        final_error = 0.0
        import time as _time
        _t_start = _time.time()
        print(f" Iniciando treinamento por {epochs} épocas...")
        for epoch in range(epochs):
            epoch_error = 0.0
            for inputs, expected in zip(X, y):
                final_output = self.predict(inputs, use_h2=self.use_h2)
                error = expected - final_output
                epoch_error += float(_np.sum(error ** 2))
                d_output = error * self._sigmoid_derivative(final_output)
                if self.use_h2:
                    error_h2 = d_output.dot(self.weights_out.T)
                    d_h2 = error_h2 * self._sigmoid_derivative(self.h2_output)
                    d_h2 = d_h2 * self.h2_mask
                    error_h1 = d_h2.dot(self.weights_h2.T)
                    d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)
                    self.weights_out += self.h2_used.reshape(-1, 1) * d_output * learning_rate
                    self.bias_out += d_output * learning_rate
                    self.weights_h2 += self.h1_output.reshape(-1, 1) * d_h2 * learning_rate
                    self.bias_h2 += d_h2 * learning_rate
                    self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                    self.bias_h1 += d_h1 * learning_rate
                else:
                    error_h1 = d_output.dot(self.weights_bypass_out.T)
                    d_h1 = error_h1 * self._sigmoid_derivative(self.h1_output)
                    self.weights_bypass_out += self.h1_output.reshape(-1, 1) * d_output * learning_rate
                    self.bias_bypass_out += d_output * learning_rate
                    self.weights_h1 += inputs.reshape(-1, 1) * d_h1 * learning_rate
                    self.bias_h1 += d_h1 * learning_rate
            final_error = epoch_error
            # Log de progresso do treino final (a cada 500 épocas e na última)
            if ((epoch + 1) % 500 == 0) or ((epoch + 1) == epochs):
                print(f"   Época {epoch+1}/{epochs}, Erro Total: {epoch_error:.10f}")
        _t_end = _time.time()
        print(f" Treinamento concluído em {_t_end - _t_start:.2f} segundos.")
        return final_error

    def train(self, X, y, preview_epochs=100, preview_lr=0.1, final_epochs=10000, final_lr=0.1, try_both=True):
        """Orquestra a prévia (50 épocas, LR variável) e o treino final (3000 épocas, LR=0.1).
        Comentário: escolhe a melhor configuração da H2 (ou bypass) com base no menor erro final da prévia.
        """
        import time as _time
        start = _time.time()
        pre_start = _time.time()
        # Pré-seleção de melhor cenário
        if try_both:
            label, prev_err, chosen_mask, chosen_snap, err_hist = self._preselect_best(X, y, epochs=preview_epochs, base_lr=preview_lr)
            pre_end = _time.time()
        else:
            chosen_snap = self._snapshot()
            label = 'current'
            prev_err, err_hist = self._train_single_variable_lr(X, y, preview_epochs, preview_lr, use_h2_flag=self.use_h2, h2_mask=self.h2_mask)
            chosen_mask = self.h2_mask.copy() if self.use_h2 else None
        # Restaurar cenário vencedor e aplicar treino final fixo
        self._restore(chosen_snap)
        final_start = _time.time()
        final_err = self._train_final_fixed(X, y, epochs=final_epochs, learning_rate=final_lr, use_h2_flag=(label != 'bypass'))
        final_end = _time.time()
        pre_dur = pre_end - pre_start
        fin_dur = final_end - final_start
        total_dur = final_end - start
        print(f" Tempo prévia: {pre_dur:.2f}s")
        print(f" Tempo treino final: {fin_dur:.2f}s")
        print(f" Tempo total: {total_dur:.2f}s")
        print(f"SmartComMLP: prévia label={label}, erro_final_prévia={prev_err:.10f}, treino_final_err={final_err:.10f}")
        return {
            'label': label,
            'preview_error': prev_err,
            'final_error': final_err,
            'use_h2': (label != 'bypass'),
            'h2_mask': (chosen_mask.copy() if chosen_mask is not None else None),
            'err_history_preview': err_hist,
        }

smart_com_mlp = SmartComMLP(initial_weights=initial_weights)
# Treinar a rede
print("--- Iniciando Treinamento ---")
smart_com_mlp.train(X_train, y_train)
print("--- Treinamento Concluído ---\n")

print("="*40)
print("      AVALIAÇÃO DO MODELO GATE")
print("="*40)

# Definir os mesmos casos de teste
test_1 = np.array([0.2, 0.8, 0.1]) # Aprovação Esperada: 1
test_2 = np.array([0.3, 0.2, 0.2]) # Reprovação Esperada: 0
test_3 = np.array([0.8, 0.9, 0.1]) # Reprovação por Temperatura (CASO CRÍTICO): 0

tests = [test_1, test_2, test_3]
expected_results = [1, 0, 0]
test_names = ["Aprovação Normal", "Reprovação Normal", "Reprovação por Temperatura (Crítico)"]

for i, test_case in enumerate(tests):
    print(f"\n--- Caso de Teste: {test_names[i]} ---")
    print(f"Entrada: {test_case}, Resultado Esperado: {expected_results[i]}")

    # Obter a predição do modelo tradicional
    prediction = smart_com_mlp.predict(test_case)
    resultado_final = 'Aprovado' if prediction > 0.5 else 'Reprovado'
    print(f"  Predição da Standard MLP: {prediction[0]:.10f} -> {resultado_final}")

--- Iniciando Treinamento ---
Época 100/100, use_h2=False, h2_false=[0, 1, 2, 3], Erro: 11.2616414507
Época 100/100, use_h2=True, h2_false=[], Erro: 0.0204506812
Época 100/100, use_h2=True, h2_false=[3], Erro: 1.2947256724
Época 100/100, use_h2=True, h2_false=[2], Erro: 0.0366747383
Época 100/100, use_h2=True, h2_false=[1], Erro: 0.0401733478
Época 100/100, use_h2=True, h2_false=[0], Erro: 0.1009119091
Época 100/100, use_h2=True, h2_false=[2, 3], Erro: 1.7264459961
Época 100/100, use_h2=True, h2_false=[1, 3], Erro: 1.8071899062
Época 100/100, use_h2=True, h2_false=[1, 2], Erro: 0.1088909881
Época 100/100, use_h2=True, h2_false=[0, 3], Erro: 1.8246813260
Época 100/100, use_h2=True, h2_false=[0, 2], Erro: 0.0865683697
Época 100/100, use_h2=True, h2_false=[0, 1], Erro: 0.0886595100
Época 100/100, use_h2=True, h2_false=[1, 2, 3], Erro: 337.9703967426
Época 100/100, use_h2=True, h2_false=[0, 2, 3], Erro: 2.5779504524
Época 100/100, use_h2=True, h2_false=[0, 1, 3], Erro: 2.7669771761
Época 1